# Clean the gold datasets

Merges the gold datasets into one CSV and cleans the code text with the project's own preprocessing (`strip_boilerplate`).

Input: `data/processed/<project>/artifacts.json`, `gold.json`. Output: `data/final/all_projects_clean.csv`.

Run the cells in order from the project root.

In [1]:
import sys, json, csv
from pathlib import Path
ROOT = Path('.').resolve()
if not (ROOT / 'agents').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from agents.traceability_agent.preprocess import strip_boilerplate
from shared.schemas.traceability import Artifact
PROCESSED = ROOT / 'data' / 'processed'
print('project root:', ROOT)

project root: /Users/habbglobal/Desktop/J26-SE-327/project


## 1. Read every dataset's artifacts and gold pairs
Only pairs labelled in `gold.json` are kept (true and false), so each row has a real answer.

In [2]:
rows, dropped = [], {'empty_after_cleaning_gold0': 0, 'empty_after_cleaning_gold1': 0}
lines_removed = 0
for folder in sorted(p for p in PROCESSED.iterdir() if (p / 'gold.json').exists()):
    arts = {a['id']: Artifact(**a) for a in json.loads((folder / 'artifacts.json').read_text(encoding='utf-8'))}
    for req_id, code_id, is_true in json.loads((folder / 'gold.json').read_text(encoding='utf-8')):
        req, code = arts[req_id], arts[code_id]
        cleaned = strip_boilerplate(code)
        if not cleaned.strip():
            dropped[f'empty_after_cleaning_gold{int(bool(is_true))}'] += 1
            continue
        lines_removed += code.text.count('\n') - cleaned.count('\n')
        rows.append([folder.name, req_id, ' '.join(req.text.split()), code_id, ' '.join(cleaned.split()), int(bool(is_true))])
print('rows', len(rows), '| dropped', dropped, '| lines removed', lines_removed)

rows 70738 | dropped {'empty_after_cleaning_gold0': 131, 'empty_after_cleaning_gold1': 0} | lines removed 819229


## 2. Write the clean CSV

In [3]:
OUT = ROOT / 'data' / 'final'
OUT.mkdir(parents=True, exist_ok=True)
with (OUT / 'all_projects_clean.csv').open('w', encoding='utf-8', newline='') as fh:
    w = csv.writer(fh)
    w.writerow(['project', 'requirement_id', 'requirement_text', 'code_id', 'code_text', 'gold'])
    w.writerows(rows)
print('written', len(rows), 'rows ->', OUT / 'all_projects_clean.csv')

written 70738 rows -> /Users/habbglobal/Desktop/J26-SE-327/project/data/final/all_projects_clean.csv


## 3. Quick check
Positives per project (should match `traces.json`).

In [4]:
import collections
c = collections.Counter((r[0], r[5]) for r in rows)
for proj in sorted({r[0] for r in rows}):
    print(f'{proj:8} gold=1: {c[(proj, 1)]:5}  gold=0: {c[(proj, 0)]:6}')

cm1      gold=1:    45  gold=0:   1121
eanci    gold=1:   567  gold=0:   1578
ebt      gold=1:    98  gold=0:   1952
etour    gold=1:   308  gold=0:   6420
itrust   gold=1:   399  gold=0:  50822
libest   gold=1:   204  gold=0:    524
smos     gold=1:  1044  gold=0:   5656
